In [1]:
import os
import re
import datetime
import pathlib

import pandas as pd
import numpy as np
import geopandas as gpd
import movingpandas as mpd
import matplotlib.pyplot as plt

import dask_geopandas as dgpd
import dask.dataframe as dd
import dask.distributed

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
src_dir = pathlib.Path('~').expanduser() / 'data/ais'

In [3]:
ark_gdf = gpd.read_file(src_dir / 'ARK.gpkg').dissolve()
ark_gdf = ark_gdf.set_crs("EPSG:28992")
ark_geometry = ark_gdf.geometry.iloc[0]

In [4]:
cluster = dask.distributed.LocalCluster(n_workers=10, memory_limit="4GB")
cluster

LocalCluster(7bf6c523, 'tcp://127.0.0.1:63219', workers=10, threads=10, memory=37.25 GiB)

2026-11-05 12:22:12,789 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 19b743c35703ce4b2eb43542b511078f initialized by task ('shuffle-transfer-19b743c35703ce4b2eb43542b511078f', 11837) executed on worker tcp://127.0.0.1:63269
2026-11-05 12:23:26,002 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 714a403953380443cee01996905cb572 initialized by task ('shuffle-transfer-714a403953380443cee01996905cb572', 18) executed on worker tcp://127.0.0.1:63246
2026-11-05 12:25:40,846 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 19b743c35703ce4b2eb43542b511078f deactivated due to stimulus 'task-finished-1793877940.8432949'
2026-11-05 12:25:57,719 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 714a403953380443cee01996905cb572 deactivated due to stimulus 'task-finished-1793877957.718463'
2026-11-05 12:35:05,066 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 339c8635858a59f2fb7e2fe269c86d9c initialized by task ('shuffle-transfer-339c863585

In [5]:
client = dask.distributed.Client(address=cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 10
Total threads: 10,Total memory: 37.25 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:63219,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:63248,Total threads: 1
Dashboard: http://127.0.0.1:63251/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:63222,


In [6]:
gddf = dgpd.read_parquet(src_dir / "filtered_ark_ais_data")
gddf.head()

,timestamp_recv,mmsi_hash,longitude,latitude,length,beam,objecttype,cog,sog,heading,shiptypeais,geom,year,month,day
3,2026-07-01 00:00:05.016000+00:00,2650cfc875cb3852f9701a08564117c1,5.03818,52.134152,110.0,10.0,VESA,132.0,7.2,131.0,80,POINT (131103.681 460718.692),2026,7,1
6,2026-07-01 00:00:11.171000+00:00,bd41feb8f33c56e4dd44c2762e54214d,5.07303,52.115548,140.0,10.0,VESA,135.0,7.4,136.0,69,POINT (133480.762 458637.931),2026,7,1
9,2026-07-01 00:00:15.234000+00:00,2650cfc875cb3852f9701a08564117c1,5.03860,52.133933,110.0,10.0,VESA,130.2,7.1,130.0,80,POINT (131132.32 460694.188),2026,7,1
11,2026-07-01 00:00:19.326000+00:00,bd41feb8f33c56e4dd44c2762e54214d,5.07334,52.115350,140.0,10.0,VESA,137.0,7.4,137.0,69,POINT (133501.9 458615.809),2026,7,1
15,2026-07-01 00:00:26.686000+00:00,2650cfc875cb3852f9701a08564117c1,5.03907,52.133690,110.0,10.0,VESA,130.3,7.1,129.0,80,POINT (131164.369 460666.998),2026,7,1


In [7]:
meta = pd.Series(dtype='datetime64[ns]')
# gddf['t'] = gddf['timestamp_recv'].apply(
#         lambda x: datetime.datetime.fromisoformat(x).replace(tzinfo=None), meta=meta
# )
gddf['t'] = gddf['timestamp_recv']
gddf['traj_id'] = gddf['mmsi_hash']
gddf['ship_length'] = gddf['length']
gddf = gddf.rename_geometry("geometry")

In [10]:
def df_to_splitted(gdf, points=True):
    # trajectorize
    traj = mpd.TrajectoryCollection(gdf, t='t', traj_id_col='traj_id', crs='EPSG:28992')

    traj_stop_splitted = mpd.ObservationGapSplitter(traj).split(gap=datetime.timedelta(minutes=10))
    
    # detect stops (very slow)
    traj_stop_splitted = mpd.StopSplitter(traj_stop_splitted).split(min_length=5, max_diameter=50, min_duration=datetime.timedelta(minutes=10))
    
    # add extra info
    traj_stop_splitted.add_acceleration(overwrite=True)
    traj_stop_splitted.add_angular_difference(overwrite=True)
    traj_stop_splitted.add_direction(overwrite=True)
    traj_stop_splitted.add_speed(overwrite=True)
    traj_stop_splitted.add_traj_id(overwrite=True)

    if points:
        final_gdf = traj_stop_splitted.to_point_gdf().reset_index()
        # final_gdf = final_gdf.rename_geometry("geometry")
    else:
        final_gdf = traj_stop_splitted.to_traj_gdf().reset_index()
        # final_gdf = final_gdf.rename_geometry("geometry")
        final_gdf["year"] = final_gdf["start_t"].dt.year
        final_gdf["month"] = final_gdf["start_t"].dt.month
        final_gdf["day"] = final_gdf["start_t"].dt.day

    return final_gdf

In [11]:
mmsi_hash_example = 'ab9a18bcc506bcd3b94fe0c52b3e0e90'
meta_points = df_to_splitted(
    gddf[gddf["mmsi_hash"] == mmsi_hash_example].compute(),
    points=True,
).iloc[:0]

meta_trajs = df_to_splitted(
    gddf[gddf["mmsi_hash"] == mmsi_hash_example].compute(),
    points=False,
).iloc[:0]


def df_to_splitted_safe(gdf, points=True):
    if gdf.empty:
        return meta_points.copy()

    traj = mpd.TrajectoryCollection(
        gdf.sort_values("t"),
        t="t",
        traj_id_col="traj_id",
        crs="EPSG:28992",
    )

    split = mpd.ObservationGapSplitter(traj).split(
        gap=datetime.timedelta(minutes=10)
    )

    split = mpd.StopSplitter(split).split(
        max_diameter=50,
        min_duration=datetime.timedelta(minutes=10),
        min_length=5,
    )

    if len(split.trajectories) == 0:
        if points:
            return meta_points.copy()
        else:
            return meta_trajs.copy()

    split.add_acceleration(overwrite=True)
    split.add_angular_difference(overwrite=True)
    split.add_direction(overwrite=True)
    split.add_speed(overwrite=True)
    split.add_traj_id(overwrite=True)
    if points:
        final_gdf = split.to_point_gdf().reset_index()
        # final_gdf = final_gdf.rename_geometry("geometry")
    else:
        final_gdf = split.to_traj_gdf().reset_index()
        # final_gdf = final_gdf.rename_geometry("geometry")
        final_gdf["year"] = final_gdf["start_t"].dt.year
        final_gdf["month"] = final_gdf["start_t"].dt.month
        final_gdf["day"] = final_gdf["start_t"].dt.day

    return final_gdf

In [12]:
# 1. Group complete trajectories together
gddf_by_traj = gddf.shuffle(
    on="traj_id",
    npartitions=100,
)

# 2. Process trajectories without computing everything into memory
gdf_points = gddf_by_traj.map_partitions(
    df_to_splitted_safe,
    points=True,
    meta=meta_points,
)

# 3. Group output by calendar day
gdf_points_by_day = gdf_points.shuffle(
    on=["year", "month", "day"],
    npartitions=100,
)

# 4. Write one partitioned dataset
gdf_points_by_day.to_parquet(
    src_dir / "ark_trajectory_points_no_stops",
    partition_on=["year", "month", "day"],
    compression="zstd",
    write_index=False,
)

/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This i

In [13]:
# 1. Group complete trajectories together
gddf_by_traj = gddf.shuffle(
    on="traj_id",
    npartitions=100,
)

# 2. Process trajectories without computing everything into memory
gdf_trajs = gddf_by_traj.map_partitions(
    df_to_splitted_safe,
    points=False,
    meta=meta_trajs,
)

# 3. Group output by calendar day
gdf_trajs_by_day = gdf_trajs.shuffle(
    on=["year", "month", "day"],
    npartitions=100,
)

# 4. Write one partitioned dataset
gdf_trajs_by_day.to_parquet(
    src_dir / "ark_trajectories_no_stops",
    partition_on=["year", "month", "day"],
    compression="zstd",
    write_index=False,
)

/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This i

In [12]:
gdf_trajs_by_day = gdf_trajs_by_day.compute()
gdf_trajs_by_day.to_file(src_dir / 'trajectories_ark.gpkg')

/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This i

In [9]:
mmsi_hash_example = 'ab9a18bcc506bcd3b94fe0c52b3e0e90'

meta_points = df_to_splitted(gddf[gddf['mmsi_hash'] == mmsi_hash_example].compute(), points=True)
gdf_points = gddf.map_partitions(df_to_splitted, points=True, meta=meta_points).compute()
gdf_points.to_parquet(src_dir / "ark_trajectory_points", partition_on=["year", "month", "day"])

meta_trajectories = df_to_splitted(gddf[gddf['mmsi_hash'] == mmsi_hash_example].compute(), points=False)
gdf_trajectories = gddf.map_partitions(df_to_splitted, points=False, meta=meta_trajectories)
gdf_trajectories.to_parquet(src_dir / "ark_trajectories", partition_on=["year", "month", "day"])

gdf_trajectories = gdf_trajectories.compute()
gdf_trajectories.to_file(src_dir / 'trajectories.gpkg')

/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/trajectory.py:162: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)
/Users/hemert/projects/scheepvaart-gedrag-ark/.venv/lib/python3.13/site-packages/movingpandas/__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings

ValueError: No objects to concatenate

In [10]:
gdf_points

NameError: name 'gdf_points' is not defined

In [11]:
meta_points

,timestamp_recv,mmsi_hash,longitude,latitude,length,beam,objecttype,cog,sog,heading,...,geom,year,month,day,traj_id,ship_length,acceleration,angular_difference,direction,speed
t,,,,,,,,,,,,,,,,,,,,,
2026-07-03 06:26:11.215,2026-07-03 06:26:11.215000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.08165,52.079138,70.0,10.0,VESA,334.0,9.8,NaN,...,POINT (134054.141 454584.485),2026,7,3,ab9a18bcc506bcd3b94fe0c52b3e0e90_0,70.0,0.000000,0.000000,340.572909,4.597575
2026-07-03 06:26:20.518,2026-07-03 06:26:20.518000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.08144,52.079500,70.0,10.0,VESA,343.0,8.5,NaN,...,POINT (134039.915 454624.821),2026,7,3,ab9a18bcc506bcd3b94fe0c52b3e0e90_0,70.0,0.000000,0.000000,340.572909,4.597575
2026-07-03 06:26:29.745,2026-07-03 06:26:29.745000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.08129,52.079855,70.0,10.0,VESA,343.9,8.5,NaN,...,POINT (134029.799 454664.361),2026,7,3,ab9a18bcc506bcd3b94fe0c52b3e0e90_0,70.0,-0.018890,5.076087,345.648996,4.423275
2026-07-03 06:26:40.157,2026-07-03 06:26:40.157000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.08113,52.080243,70.0,10.0,VESA,345.6,8.6,NaN,...,POINT (134019.013 454707.576),2026,7,3,ab9a18bcc506bcd3b94fe0c52b3e0e90_0,70.0,-0.013975,0.336781,345.985777,4.277769
2026-07-03 06:26:50.891,2026-07-03 06:26:50.891000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.08097,52.080663,70.0,10.0,VESA,347.3,8.5,NaN,...,POINT (134008.242 454754.351),2026,7,3,ab9a18bcc506bcd3b94fe0c52b3e0e90_0,70.0,0.018063,1.046830,347.032608,4.471659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-07 16:15:36.751,2026-10-07 16:15:36.751000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.07590,52.104800,70.0,10.0,VESA,342.3,3.1,NaN,...,POINT (133672.215 457441.274),2026,10,7,ab9a18bcc506bcd3b94fe0c52b3e0e90_26,70.0,0.135940,5.227967,351.493873,2.339125
2026-10-07 16:15:46.985,2026-10-07 16:15:46.985000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.07582,52.104900,70.0,10.0,VESA,337.5,2.8,NaN,...,POINT (133666.782 457452.424),2026,10,7,ab9a18bcc506bcd3b94fe0c52b3e0e90_26,70.0,-0.110144,17.474190,334.019682,1.211915
2026-10-07 16:15:55.689,2026-10-07 16:15:55.689000+00:00,ab9a18bcc506bcd3b94fe0c52b3e0e90,5.07573,52.105000,70.0,10.0,VESA,333.2,2.7,NaN,...,POINT (133660.663 457463.576),2026,10,7,ab9a18bcc506bcd3b94fe0c52b3e0e90_26,70.0,0.028668,2.769222,331.250460,1.461437
